# Full hallucination-autocorrelation dissertation experiment

This notebook runs the complete executable methodology on real model generations: four task types, two model families at two scales, two temperatures, two prompt structures, HHEM sentence annotation, formal temporal tests, token-feature warning models, and held-out truncate-and-regenerate interventions.

**Before running:** choose a GPU accelerator and enable Internet in Kaggle. Upload `hallucination_autocorrelation_project.zip` as a private Kaggle Dataset attached to this notebook.

In [17]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

working = Path('/kaggle/working')
input_root = Path('/kaggle/input')
direct_scripts = list(input_root.rglob('run_model_experiment.py'))
if direct_scripts:
    PROJECT = direct_scripts[0].parent
else:
    archives = list(input_root.rglob('hallucination_autocorrelation_project.zip'))
    if not archives:
        raise FileNotFoundError('Attach hallucination_autocorrelation_project.zip as a Kaggle Dataset.')
    PROJECT = working / 'hallucination_autocorrelation_project'
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(PROJECT)
print('Project:', PROJECT)
assert (PROJECT / 'run_model_experiment.py').exists()

Project: /kaggle/input/datasets/maryamanwer/hallucination-autocorrelation-project


In [18]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT / 'requirements-kaggle.txt')
], check=True)
print('Dependencies installed. Restart the kernel only if Kaggle explicitly requests it.')

Dependencies installed. Restart the kernel only if Kaggle explicitly requests it.


In [19]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing.'
print('GPU:', torch.cuda.get_device_name(0))




# Full factorial, Kaggle-feasible configuration. Increase samples/replicates
# for the final dissertation run; checkpointing allows the generation cell
# to resume safely within a session.
SEED = 42
SAMPLES_PER_TASK = 5
MODELS = [
    'HuggingFaceTB/SmolLM2-360M-Instruct',
    'HuggingFaceTB/SmolLM2-1.7B-Instruct',
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-7B-Instruct',
]
TEMPERATURES = [0.3, 0.9]
REPLICATES = 1
MAX_NEW_TOKENS = 128
PERMUTATIONS = 99




RUN_ROOT = working / 'hallucination_fast_run'
BENCHMARK = RUN_ROOT / 'benchmark.jsonl'
BASELINE = RUN_ROOT / 'baseline'
ANALYSIS = RUN_ROOT / 'analysis'
INTERVENTION = RUN_ROOT / 'intervention'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Expected baseline generations:', SAMPLES_PER_TASK * 4 * 2 * len(MODELS) * len(TEMPERATURES) * REPLICATES)

GPU: Tesla T4
Expected baseline generations: 320


## 1. Construct the four-task controlled benchmark

Sources are SQuAD/Wikipedia for biographical QA and long-form generation, SciTLDR for scientific summarisation, and HotpotQA for multi-hop reasoning. Every source receives strict-grounding and claim-verification prompt variants.

In [20]:
subprocess.run([
    sys.executable, str(PROJECT / 'build_benchmark.py'),
    '--output', str(BENCHMARK),
    '--samples-per-task', str(SAMPLES_PER_TASK),
    '--seed', str(SEED),
], check=True)

Wrote 40 prompt records to /kaggle/working/hallucination_fast_run/benchmark.jsonl
{
  "biographical_qa": 10,
  "long_form_generation": 10,
  "multi_hop_reasoning": 10,
  "scientific_summarisation": 10
}


CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/maryamanwer/hallucination-autocorrelation-project/build_benchmark.py', '--output', '/kaggle/working/hallucination_fast_run/benchmark.jsonl', '--samples-per-task', '5', '--seed', '42'], returncode=0)

## 2. Generate and annotate the baseline corpus

Models are loaded sequentially in 4-bit NF4. Each output records normalized token log probabilities, token entropy, local coherence and HHEM source-support labels. The command checkpoints every generation and skips completed IDs when rerun.

In [21]:
command = [
    sys.executable, '-u', str(PROJECT / 'run_model_experiment.py'),
    '--benchmark', str(BENCHMARK),
    '--models', *MODELS,
    '--temperatures', *[str(value) for value in TEMPERATURES],
    '--replicates', str(REPLICATES),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--output-dir', str(BASELINE),
    '--seed', str(SEED),
    '--load-in-4bit', '--resume'
]
subprocess.run(command, check=True)

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


Loading generation model: HuggingFaceTB/SmolLM2-360M-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


  source=bio:572906e23f37b31900477f8e::strict_grounding temp=0.3 replicate=0 sentences=5
  source=bio:572906e23f37b31900477f8e::strict_grounding temp=0.9 replicate=0 sentences=1
  source=bio:572906e23f37b31900477f8e::claim_verification temp=0.3 replicate=0 sentences=5
  source=bio:572906e23f37b31900477f8e::claim_verification temp=0.9 replicate=0 sentences=5
  source=bio:57274d905951b619008f87e3::strict_grounding temp=0.3 replicate=0 sentences=1
  source=bio:57274d905951b619008f87e3::strict_grounding temp=0.9 replicate=0 sentences=5
  source=bio:57274d905951b619008f87e3::claim_verification temp=0.3 replicate=0 sentences=1
  source=bio:57274d905951b619008f87e3::claim_verification temp=0.9 replicate=0 sentences=3
  source=bio:5730b255396df919000962b4::strict_grounding temp=0.3 replicate=0 sentences=4
  source=bio:5730b255396df919000962b4::strict_grounding temp=0.9 replicate=0 sentences=3
  source=bio:5730b255396df919000962b4::claim_verification temp=0.3 replicate=0 sentences=3
  source=bi

Token indices sequence length is longer than the specified maximum sequence length for this model (1642 > 512). Running this sequence through the model will result in indexing errors


  source=multihop:5a8810485542997e5c09a590::strict_grounding temp=0.3 replicate=0 sentences=1
  source=multihop:5a8810485542997e5c09a590::strict_grounding temp=0.9 replicate=0 sentences=1
  source=multihop:5a8810485542997e5c09a590::claim_verification temp=0.3 replicate=0 sentences=1
  source=multihop:5a8810485542997e5c09a590::claim_verification temp=0.9 replicate=0 sentences=1
  source=multihop:5a7a96b055429941d65f26d7::strict_grounding temp=0.3 replicate=0 sentences=2
  source=multihop:5a7a96b055429941d65f26d7::strict_grounding temp=0.9 replicate=0 sentences=2
  source=multihop:5a7a96b055429941d65f26d7::claim_verification temp=0.3 replicate=0 sentences=2
  source=multihop:5a7a96b055429941d65f26d7::claim_verification temp=0.9 replicate=0 sentences=1
  source=multihop:5a88016b5542997e5c09a58a::strict_grounding temp=0.3 replicate=0 sentences=3
  source=multihop:5a88016b5542997e5c09a58a::strict_grounding temp=0.9 replicate=0 sentences=1
  source=multihop:5a88016b5542997e5c09a58a::claim_ve

Loading checkpoint shards: 100%|██████████| 4/4 [00:45<00:00, 11.43s/it]


  source=bio:572906e23f37b31900477f8e::strict_grounding temp=0.3 replicate=0 sentences=1
  source=bio:572906e23f37b31900477f8e::strict_grounding temp=0.9 replicate=0 sentences=1
  source=bio:572906e23f37b31900477f8e::claim_verification temp=0.3 replicate=0 sentences=1
  source=bio:572906e23f37b31900477f8e::claim_verification temp=0.9 replicate=0 sentences=1
  source=bio:57274d905951b619008f87e3::strict_grounding temp=0.3 replicate=0 sentences=2
  source=bio:57274d905951b619008f87e3::strict_grounding temp=0.9 replicate=0 sentences=2
  source=bio:57274d905951b619008f87e3::claim_verification temp=0.3 replicate=0 sentences=2
  source=bio:57274d905951b619008f87e3::claim_verification temp=0.9 replicate=0 sentences=2
  source=bio:5730b255396df919000962b4::strict_grounding temp=0.3 replicate=0 sentences=2
  source=bio:5730b255396df919000962b4::strict_grounding temp=0.9 replicate=0 sentences=2
  source=bio:5730b255396df919000962b4::claim_verification temp=0.3 replicate=0 sentences=2
  source=bi

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/input/datasets/maryamanwer/hallucination-autocorrelation-project/run_model_experiment.py', '--benchmark', '/kaggle/working/hallucination_fast_run/benchmark.jsonl', '--models', 'HuggingFaceTB/SmolLM2-360M-Instruct', 'HuggingFaceTB/SmolLM2-1.7B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', '--temperatures', '0.3', '0.9', '--replicates', '1', '--max-new-tokens', '128', '--output-dir', '/kaggle/working/hallucination_fast_run/baseline', '--seed', '42', '--load-in-4bit', '--resume'], returncode=0)

## 3. Formal statistics, classifier and baseline task metrics

In [22]:
subprocess.run([
    sys.executable, '-u', str(PROJECT / 'run_analysis.py'),
    '--input', str(BASELINE / 'sentence_labels.csv'),
    '--output-dir', str(ANALYSIS),
    '--dataset-name', 'Controlled four-task Kaggle generation corpus',
    '--max-lag', '5', '--permutations', str(PERMUTATIONS),
    '--seed', str(SEED),
], check=True)
subprocess.run([
    sys.executable, str(PROJECT / 'evaluate_corpus.py'),
    '--sentences', str(BASELINE / 'sentence_labels.csv'),
    '--benchmark', str(BENCHMARK),
    '--output-dir', str(ANALYSIS),
], check=True)

Analysed 320 generations.
Pooled lag-1 ACF: 0.3712
Permutation p: 0.93
Classifier ROC AUC: 0.7053
Results: /kaggle/working/hallucination_fast_run/analysis
                              model                task_type  temperature   prompt_structure  generations  factuality_rate  task_completion_score  token_f1  rouge_l  format_completion
HuggingFaceTB/SmolLM2-1.7B-Instruct          biographical_qa          0.3 claim_verification            5         0.800000               0.865000  0.178433 0.178433           0.550000
HuggingFaceTB/SmolLM2-1.7B-Instruct          biographical_qa          0.3   strict_grounding            5         0.860000               0.785000  0.103204 0.103204           0.750000
HuggingFaceTB/SmolLM2-1.7B-Instruct          biographical_qa          0.9 claim_verification            5         0.733333               0.570000  0.157273 0.157273           0.500000
HuggingFaceTB/SmolLM2-1.7B-Instruct          biographical_qa          0.9   strict_grounding            5    

CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/maryamanwer/hallucination-autocorrelation-project/evaluate_corpus.py', '--sentences', '/kaggle/working/hallucination_fast_run/baseline/sentence_labels.csv', '--benchmark', '/kaggle/working/hallucination_fast_run/benchmark.jsonl', '--output-dir', '/kaggle/working/hallucination_fast_run/analysis'], returncode=0)

## 4. Execute held-out truncate-and-regenerate interventions

Only held-out source groups are used. At the first warning, the pipeline retains the verified prefix, adds the grounding instruction, regenerates with the same model and temperature, re-runs HHEM, and compares factuality, task completion and length.

In [23]:
subprocess.run([
    sys.executable, '-u', str(PROJECT / 'run_intervention_experiment.py'),
    '--benchmark', str(BENCHMARK),
    '--baseline-csv', str(BASELINE / 'sentence_labels.csv'),
    '--classifier', str(ANALYSIS / 'early_warning_classifier.joblib'),
    '--classifier-metrics', str(ANALYSIS / 'classifier_metrics.json'),
    '--output-dir', str(INTERVENTION),
    '--evaluation-split', 'test',
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--seed', str(SEED + 1000), '--load-in-4bit'
], check=True)
subprocess.run([
    sys.executable, str(PROJECT / 'evaluate_corpus.py'),
    '--sentences', str(INTERVENTION / 'intervened_sentence_labels.csv'),
    '--benchmark', str(BENCHMARK),
    '--output-dir', str(INTERVENTION),
], check=True)

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


Loading generation model for intervention: HuggingFaceTB/SmolLM2-1.7B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Token indices sequence length is longer than the specified maximum sequence length for this model (1644 > 512). Running this sequence through the model will result in indexing errors


  generation=3ad6fcea-db8f-5644-adc8-243d32ea177b trigger=1 baseline_h=1 intervened_h=0
  generation=4234293e-12ba-5460-8041-23b657083e8f trigger=1 baseline_h=0 intervened_h=1
  generation=776de79d-651d-5685-8e16-1bedaef0c6c2 trigger=0 baseline_h=5 intervened_h=4
  generation=ac911921-3264-54ce-acb2-a29126c6a6cc trigger=0 baseline_h=1 intervened_h=0
  generation=b8b03061-1a7b-5851-8cde-55eccb2a05cc trigger=1 baseline_h=0 intervened_h=0
  generation=cffb1311-1b8f-5e74-b127-d32af1d86283 trigger=0 baseline_h=2 intervened_h=2
  generation=d3ea2234-5546-5f01-af0d-b19491432d85 trigger=0 baseline_h=4 intervened_h=2
  generation=f1dd3e0e-daac-5bd1-8233-c3fd06951e54 trigger=0 baseline_h=5 intervened_h=0
  generation=f92bef5f-93d9-5bb9-b67e-efa100b97953 trigger=0 baseline_h=1 intervened_h=0
Loading generation model for intervention: HuggingFaceTB/SmolLM2-360M-Instruct
  generation=18f34b43-7239-5711-9f1c-2703213bbbfc trigger=0 baseline_h=1 intervened_h=0
  generation=3fad1251-462e-5df9-80b9-3122

Loading checkpoint shards: 100%|██████████| 4/4 [01:07<00:00, 16.87s/it]


  generation=5111fe4d-feff-540f-ba22-a45195f66471 trigger=0 baseline_h=1 intervened_h=1
  generation=7a92efce-82e3-5e98-9b39-887eff171e17 trigger=0 baseline_h=0 intervened_h=0
  generation=b6a096dd-6059-5f89-a8a3-87521d5e5eea trigger=0 baseline_h=2 intervened_h=0
  generation=c5f54370-864c-5429-bfa9-c97092da628a trigger=0 baseline_h=0 intervened_h=1
  generation=dbbf6254-fc8f-5041-a51d-8feee19bc1f8 trigger=0 baseline_h=1 intervened_h=0
  generation=dbcb626f-9cda-5391-bb50-744018a14436 trigger=0 baseline_h=0 intervened_h=0
  generation=dd05ee66-bf3f-520a-990d-66ce6fa55588 trigger=0 baseline_h=0 intervened_h=1
  generation=e294ba85-4abd-5e5e-8288-d1237a022574 trigger=0 baseline_h=1 intervened_h=1
  generation=edab372a-b74a-5c41-aa3e-958ced1a62ec trigger=1 baseline_h=1 intervened_h=0
Intervention results: /kaggle/working/hallucination_fast_run/intervention
                              model                task_type  temperature   prompt_structure  generations  factuality_rate  task_compl

CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/maryamanwer/hallucination-autocorrelation-project/evaluate_corpus.py', '--sentences', '/kaggle/working/hallucination_fast_run/intervention/intervened_sentence_labels.csv', '--benchmark', '/kaggle/working/hallucination_fast_run/benchmark.jsonl', '--output-dir', '/kaggle/working/hallucination_fast_run/intervention'], returncode=0)

## 5. Create the required manual-verification sample

Download `manual_audit.csv`, judge each sentence against its displayed source, enter `0` or `1` in `human_hallucination`, upload the completed file, and run `manual_audit.py --score`. This human judgement cannot be truthfully automated.

In [24]:
AUDIT = RUN_ROOT / 'manual_audit.csv'
subprocess.run([
    sys.executable, str(PROJECT / 'manual_audit.py'),
    '--input', str(BASELINE / 'sentence_labels.csv'),
    '--benchmark', str(BENCHMARK),
    '--output', str(AUDIT),
    '--per-cell', '3', '--seed', str(SEED),
], check=True)
print('Manual audit file:', AUDIT)

Created 94-sentence audit at /kaggle/working/hallucination_fast_run/manual_audit.csv. Fill human_hallucination with 0 or 1, then rerun with --score.
Manual audit file: /kaggle/working/hallucination_fast_run/manual_audit.csv


## 6. Inspect and package results

In [25]:
import json, pandas as pd
from IPython.display import Markdown, display

display(Markdown((ANALYSIS / 'analysis_report.md').read_text()))
print('\nIntervention summary:')
print(json.dumps(json.loads((INTERVENTION / 'intervention_summary.json').read_text()), indent=2))
display(pd.read_csv(ANALYSIS / 'task_metrics_summary.csv').head(20))

archive = shutil.make_archive(str(working / 'hallucination_full_results'), 'zip', RUN_ROOT)
print('Download this archive from the Kaggle Output panel:', archive)

# Hallucination temporal-structure analysis

## Analysis status

Dataset: **Controlled four-task Kaggle generation corpus**.

Results use the supplied sentence-level annotations. If labels were automatically generated, validate a stratified sample manually before drawing dissertation-level conclusions.

## Dataset

- Eligible generations: 320
- Source records: 40
- Sentence labels: 1,157
- Models: HuggingFaceTB/SmolLM2-1.7B-Instruct, HuggingFaceTB/SmolLM2-360M-Instruct, Qwen/Qwen2.5-1.5B-Instruct, Qwen/Qwen2.5-7B-Instruct
- Task types: biographical_qa, long_form_generation, multi_hop_reasoning, scientific_summarisation
- Median generation length: 4.0 sentences
- Sentence-level hallucination prevalence: 0.2645

## Primary autocorrelation result

- Pooled lag-1 ACF: 0.3712
- P(next hallucination | current faithful): 0.1544
- P(next hallucination | current hallucination): 0.5175
- Persistence risk ratio: 3.35×
- Mean demeaned Durbin–Watson statistic: 1.9047
- Pooled Ljung–Box Q(5): 534.81
- Asymptotic p-value: 2.43e-113
- Within-generation permutation p-value (99 permutations): 0.9300
- Mean hallucination burst length: 1.6277 sentences
- Maximum observed burst length: 6 sentences

The prespecified positive-clustering criterion was not met in this corpus.

## Group comparisons

0 experimental-factor groups remain significant at FDR < 0.05.

| Dimension | Group | N generations | Hallucination rate | ACF(1) | Mean burst | Permutation p (FDR) |
|---|---:|---:|---:|---:|---:|---:|
| model | HuggingFaceTB/SmolLM2-1.7B-Instruct | 80 | 0.2383 | 0.4224 | 1.6512 | 1.0000 |
| model | HuggingFaceTB/SmolLM2-360M-Instruct | 80 | 0.2349 | 0.4837 | 1.7073 | 1.0000 |
| model | Qwen/Qwen2.5-1.5B-Instruct | 80 | 0.4103 | 0.2585 | 1.7778 | 1.0000 |
| model | Qwen/Qwen2.5-7B-Instruct | 80 | 0.1486 | 0.0953 | 1.1562 | 1.0000 |
| model_family | Qwen | 160 | 0.2941 | 0.2901 | 1.5865 | 1.0000 |
| model_family | SmolLM | 160 | 0.2366 | 0.4525 | 1.6786 | 1.0000 |
| task_type | biographical_qa | 80 | 0.2204 | 0.5427 | 1.5185 | 1.0000 |
| task_type | long_form_generation | 80 | 0.4515 | 0.4599 | 2.1585 | 1.0000 |
| task_type | multi_hop_reasoning | 80 | 0.2105 | 0.0226 | 1.1429 | 1.0000 |
| task_type | scientific_summarisation | 80 | 0.1234 | -0.0270 | 1.0909 | 1.0000 |
| model_scale_b | 0.36 | 80 | 0.2349 | 0.4837 | 1.7073 | 1.0000 |
| model_scale_b | 1.5 | 80 | 0.4103 | 0.2585 | 1.7778 | 1.0000 |
| model_scale_b | 1.7 | 80 | 0.2383 | 0.4224 | 1.6512 | 1.0000 |
| model_scale_b | 7.0 | 80 | 0.1486 | 0.0953 | 1.1562 | 1.0000 |
| temperature | 0.3 | 160 | 0.2297 | 0.4366 | 1.7051 | 1.0000 |
| temperature | 0.9 | 160 | 0.2993 | 0.3090 | 1.5727 | 1.0000 |
| prompt_structure | claim_verification | 160 | 0.2633 | 0.3580 | 1.5851 | 1.0000 |
| prompt_structure | strict_grounding | 160 | 0.2657 | 0.3833 | 1.6702 | 1.0000 |

## Early-warning classifier

The logistic classifier predicts whether the remaining generation contains a hallucination, using only information available at the current sentence boundary. Source groups, rather than individual responses, define the split to prevent source leakage.

- ROC AUC: 0.7053
- Average precision: 0.5991
- Precision: 0.5135
- Recall: 0.7600
- F1: 0.6129
- Tuned decision threshold: 0.2874
- Train/test examples: 649 / 188

## Intervention-trigger evaluation

- Generations triggering an intervention: 38 (0.7308)
- Trigger precision for any later hallucination: 0.5263
- Future hallucination-sentence coverage: 0.6349
- Mean warning lead when successful: 1.5000 sentences
- Mean completion retained under truncation: 0.4560

## Statistical interpretation and limitations

- The pooled ACF uses only within-response lagged pairs. Global centring avoids the finite-sample negative bias caused by separately demeaning very short binary responses.
- The randomisation test preserves each response's length and positive count while destroying order, so response-level prevalence differences remain present under the null. Group p-values are Benjamini–Hochberg adjusted.
- Durbin–Watson is descriptive here because binary labels are not regression residuals; the Ljung–Box and randomisation results are primary.
- Sentence labels inherit boundary decisions when a human span crosses a sentence boundary. A manual boundary audit should be reported.
- RAGTruth covers QA, summarisation and data-to-text. Claims about biographical QA, scientific summarisation, multi-hop reasoning or new model families require running the included generation pipeline on those tasks.
- Trigger coverage is not a counterfactual factuality improvement. A restart/steering claim requires generating and re-annotating the intervened output.
- The maximum tested lag was 5 sentences.



Intervention summary:
{
  "warning_threshold": 0.2874183521733535,
  "generations": 64,
  "triggered_generations": 38,
  "trigger_rate": 0.59375,
  "triggered_baseline_hallucination_rate": 0.32972972972972975,
  "triggered_intervened_hallucination_rate": 0.25252525252525254,
  "mean_length_ratio_triggered": 1.1176691729323307,
  "mean_baseline_task_completion_triggered": 0.39814812528005644,
  "mean_intervened_task_completion_triggered": 0.3778895135759123,
  "mean_task_completion_change_triggered": -0.020258611704144187,
  "annotation_model": "vectara/hallucination_evaluation_model",
  "synthetic_data_used": false
}


,model,task_type,temperature,prompt_structure,generations,factuality_rate,task_completion_score,token_f1,rouge_l,format_completion
0,HuggingFaceTB/SmolLM2-1.7B-Instruct,biographical_qa,0.3,claim_verification,5,0.800000,0.865000,0.178433,0.178433,0.550000
1,HuggingFaceTB/SmolLM2-1.7B-Instruct,biographical_qa,0.3,strict_grounding,5,0.860000,0.785000,0.103204,0.103204,0.750000
2,HuggingFaceTB/SmolLM2-1.7B-Instruct,biographical_qa,0.9,claim_verification,5,0.733333,0.570000,0.157273,0.157273,0.500000
3,HuggingFaceTB/SmolLM2-1.7B-Instruct,biographical_qa,0.9,strict_grounding,5,0.700000,0.615000,0.099901,0.099901,0.650000
4,HuggingFaceTB/SmolLM2-1.7B-Instruct,long_form_generation,0.3,claim_verification,5,0.520000,0.351108,0.395908,0.244440,0.600000
5,HuggingFaceTB/SmolLM2-1.7B-Instruct,long_form_generation,0.3,strict_grounding,5,0.466667,0.321217,0.371083,0.191024,0.625000
6,HuggingFaceTB/SmolLM2-1.7B-Instruct,long_form_generation,0.9,claim_verification,5,0.350000,0.317195,0.355218,0.206708,0.575000
7,HuggingFaceTB/SmolLM2-1.7B-Instruct,long_form_generation,0.9,strict_grounding,5,0.433333,0.320886,0.379598,0.244123,0.500000
8,HuggingFaceTB/SmolLM2-1.7B-Instruct,multi_hop_reasoning,0.3,claim_verification,5,0.900000,0.504872,0.120581,0.120581,0.600000
9,HuggingFaceTB/SmolLM2-1.7B-Instruct,multi_hop_reasoning,0.3,strict_grounding,5,0.871429,0.521875,0.080874,0.080874,0.733333


Download this archive from the Kaggle Output panel: /kaggle/working/hallucination_full_results.zip
